In [1]:
# Processing data and configurations

import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.ops import cascaded_union
import os

from datetime import datetime
import glob

In [2]:
def preprocess_rean(xdf):
    """Adjust longitude bounds and round coordinates."""
    xdf['longitude'] = np.round(xdf.longitude, 3)
    xdf['latitude'] = np.round(xdf.latitude, 3)
    
    if xdf.longitude.values.min() >= 0:
        xdf = roll_xdf(xdf)
    return xdf

def roll_xdf(xdf):
    """Roll longitude from 0-360 to -180-180."""
    xdf = xdf.assign_coords(longitude=(((xdf.longitude + 180) % 360) - 180))
    return xdf

def get_geometries(cut_on, shapefile=None):
    """Retrieve geometries for given region or bounding box coordinates."""
    if cut_on == 'europe':
        geometries = [-25, 32, 48.5, 75]
    elif cut_on == 'SADC':
        geometries = [2, -37, 60, 7]
    elif isinstance(cut_on, list):
        geometries = cut_on
    else:   
        if not shapefile:
            geometries = get_boundaries(cut_on.capitalize(), continent=False)
        else:
            geometries = gpd.read_file(shapefile)
            geometries = geometries[geometries['NAME_EN'] == cut_on]['geometry']
    return geometries

def get_boundaries(name, continent=False):
    """Retrieve boundary geometries by name."""
    world_path = '../world_map/'
    world = gpd.read_file(world_path)
    
    if continent:
        world = world.query('REGION_UN == @name')
        boundary = gpd.GeoSeries(cascaded_union(world['geometry']))
    elif name in world['NAME_EN'].values:
        region_mask = world[world['NAME_EN'] == name]
        polygons = region_mask['geometry']
        boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)
    return boundary


In [3]:
def load_data(data_path):
    """Load river discharge dataset based on specified variable."""
    full_path = f"{data_path}/"
    files = sorted(glob.glob(full_path + 'surface_solar_radiation_*.nc'))
    print(files)
    try:
        xdf = xr.open_mfdataset(files, engine='netcdf4', preprocess=preprocess_rean)
    except IOError as e:
        raise IOError(f"Files not found at {data_path}. Error: {e}")
    return xdf


def include_time_coord(ds):
    """Ensure time coordinate inclusion."""
    if 'time' not in ds.coords:
        ds = ds.assign_coords(time=('time', pd.date_range(datetime.now(), periods=ds.dims['time'])))
    return ds

def normalize_naming(xdf):
    """Standardize variable names as required."""
    # Placeholder normalization, as naming conventions may vary
    return xdf.rename({'var_name_placeholder': 'variable_standard_name'}) 

def aggregate_daymean(xdf, var_name):
    """Compute daily mean for given variable."""
    xdf = xdf.rename({'valid_time':'time'})
    xdf_daymean = xdf[var_name].sel(time=xdf.time.dt.hour == 0) / 24
    return xdf_daymean

def cut_xdf(xdf, geometries):
    
    xdf = xdf.sortby('latitude')
    """Subset xdf using provided geometries."""
    # Assuming geometries are bounding box [lon_min, lat_min, lon_max, lat_max]
    # Assuming 'geometries' is a DataFrame with bounding box columns
    lon_min = geometries.bounds['minx'].values[0]  # Extract minimum longitude
    lat_min = geometries.bounds['miny'].values[0]  # Extract minimum latitude
    lon_max = geometries.bounds['maxx'].values[0]  # Extract maximum longitude
    lat_max = geometries.bounds['maxy'].values[0]  # Extract maximum latitude
    
    xdf_cut = xdf.sel(longitude=slice(lon_min, lon_max), latitude=slice(lat_min, lat_max))
    return xdf_cut

In [4]:
country = 'croatia' ## FILL WITH THE COUNTRY NAME e.g. croatia, cuba

# Processing data and configurations
shapefile_path = '../world_map/ne_50m_admin_0_countries.shp'
geometries = get_geometries(f'{country.capitalize()}', shapefile_path)

output_path = f'../data/processed_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/preprocess_conda/share/proj failed


### Preprocessing of the CDS reanalysis

In [5]:
## 
data_path = '../data/cds_downloads/' ## FILL WITH THE PATH of ERA5LAND data

In [6]:
def main(variable, data_sources, start_date, end_date, loading_area, aggregation_mode, overwrite=False, data_path='', output_path=''):
    # Step 1: Define date range and load data
    xdf = load_data(data_path=data_path)
    
    # Step 2: Apply geographic masking if specified
    if loading_area:
        geometries = get_geometries(loading_area)
        xdf = cut_xdf(xdf, geometries)
        print(f"Data masked to the specified loading area: {loading_area}")

    # Step 3: Aggregate data by daily mean if required
    if aggregation_mode == 'daymean':
        xdf = aggregate_daymean(xdf, var_name=variable)
        print("Data aggregated by daily mean.")

    # Step 4: Save output if overwrite is enabled
    output_filename = f"{output_path}{variable}_{start_date}_{end_date}_processed.nc"
    if overwrite or not os.path.exists(output_filename):
        os.makedirs(output_path, exist_ok=True)
        print(xdf)
        xdf = xdf.drop('number')
        xdf.to_netcdf(output_filename)
        print(f"Data written to {output_filename}")
    else:
        print(f"File already exists and overwrite is set to False: {output_filename}")

# Parameters (Example Usage)
main(
    variable='ssrd',
    data_sources='era5land',
    start_date='1992-01',
    end_date='2020-12',
    loading_area=country,
    aggregation_mode='daymean',
    data_path=data_path,
    output_path=output_path,
    overwrite=True
)

['../data/cds_downloads/surface_solar_radiation_downwards_1992_01.nc']


/tmp/ipykernel_1340696/2069445547.py:42: ShapelyDeprecationWarning: The 'cascaded_union()' function is deprecated. Use 'unary_union()' instead.
  boundary = gpd.GeoSeries(cascaded_union(polygons)) if (polygons.geom_type == 'MultiPolygon').any() else gpd.GeoSeries(polygons)
/tmp/ipykernel_1340696/346420892.py:21: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  xdf = xdf.drop('number')


Data masked to the specified loading area: croatia
Data aggregated by daily mean.
<xarray.DataArray 'ssrd' (time: 31, latitude: 41, longitude: 59)> Size: 300kB
dask.array<truediv, shape=(31, 41, 59), dtype=float32, chunksize=(31, 41, 59), chunktype=numpy.ndarray>
Coordinates:
    number     int64 8B ...
  * time       (time) datetime64[ns] 248B 1992-01-01 1992-01-02 ... 1992-01-31
  * latitude   (latitude) float64 328B 42.5 42.6 42.7 42.8 ... 46.3 46.4 46.5
    expver     (time) <U4 496B dask.array<chunksize=(31,), meta=np.ndarray>
  * longitude  (longitude) float64 472B 13.6 13.7 13.8 13.9 ... 19.2 19.3 19.4
Data written to ../data/processed_data/croatia/ssrd_1992-01_1992-01_processed.nc


### Preprocessing of the SIS from satellite data
Croatia use case. The SARAH-3 does not cover the longitudes of Cuba 

In [5]:
sis_path = f'../data/dwd_downloads/' ## FILL WITH THE PATH of Satellite Data

In [6]:
def preprocess_sat(sis):
    sis = sis.rename({'lon':'longitude','lat':'latitude'}).SIS.to_dataset()
    return sis

In [7]:
sis = xr.open_mfdataset(sis_path+'*.nc', preprocess=preprocess_sat)

In [8]:
sis_months = sis.groupby('time.month').mean()

In [9]:
sis_country = cut_xdf(sis_months, geometries)

In [ ]:
sis_country.to_netcdf(output_path+'sis_months_1990_2019_processed.nc') 